In [1]:
import duckdb
con = duckdb.connect("data/flights.duckdb")

## **Delay and Cancellation Rate by Airline Name**

In [67]:
by_name = con.execute('''SELECT carrier_name, COUNT(*) AS num_flights, AVG(delayed) AS
     avg_delay_rate, AVG(cancelled) AS cancellation_rate FROM fact_flights f LEFT JOIN dim_carrier c ON f.carrier_code = c.carrier_code
     GROUP BY carrier_name ORDER BY avg_delay_rate DESC''').df()
display(by_name)
assert(by_name['num_flights'].sum() == 7043858)



,carrier_name,num_flights,avg_delay_rate,cancellation_rate
0,JetBlue Airways,237090,0.269698,0.030558
1,Spirit Air Lines,117136,0.265603,0.039390
2,Frontier Airlines Inc.,209405,0.262201,0.019226
3,Allegiant Air,126499,0.253253,0.006593
4,PSA Airlines Inc.,242215,0.252507,0.042904
5,American Airlines Inc.,987645,0.247469,0.021971
6,Southwest Airlines Co.,1389951,0.227284,0.008805
7,Republic Airline,363168,0.207302,0.041333
8,SkyWest Airlines Inc.,854130,0.203930,0.017826
9,United Air Lines Inc.,825693,0.200620,0.010306


JetBlue Airways still bodes the highest average delay rate and top 4 in cancellation rate. If you take a combination of both of them into account you may say that Spirit Air Lines or PSA Airlines inc.

## **Delay and Cancellation Rate by Season**

In [66]:
by_season = con.execute('''SELECT season, COUNT(*) AS num_flights, AVG(delayed) AS
     avg_delay_rate, AVG(cancelled) AS cancellation_rate FROM fact_flights LEFT JOIN dim_date ON flight_date = date
     GROUP BY season ORDER BY avg_delay_rate DESC''').df()
display(by_season)
assert(by_season['num_flights'].sum() == 7043858)

,season,num_flights,avg_delay_rate,cancellation_rate
0,Summer,1841925,0.248522,0.017933
1,Winter,1641344,0.219111,0.028173
2,Spring,1821756,0.212856,0.015822
3,Fall,1738833,0.185555,0.011666


Summer had the highest delay rate at 25%, Fall the lowest at 19%.

## **Delay and Cancellation Rate by Holiday**

In [68]:
by_holiday = con.execute('''SELECT is_holiday, COUNT(*) AS num_flights, AVG(delayed) AS
     avg_delay_rate, AVG(cancelled) AS cancellation_rate FROM fact_flights LEFT JOIN dim_date ON flight_date = date
     GROUP BY is_holiday ORDER BY avg_delay_rate DESC''').df()
display(by_holiday)
assert(by_holiday['num_flights'].sum() == 7043858)

,is_holiday,num_flights,avg_delay_rate,cancellation_rate
0,True,204296,0.233288,0.020544
1,False,6839562,0.216334,0.018157


## T-Test

## **Delay and Cancellation Rate by Routes**

In [85]:
by_route = con.execute('''SELECT o.city AS origin_city, d.city AS destination_city, origin_airport, dest_airport, COUNT(*) AS num_flights, AVG(delayed) AS
     avg_delay_rate, AVG(cancelled) AS cancellation_rate FROM fact_flights f LEFT JOIN dim_airport o ON f.origin_airport = o.airport_code
     LEFT JOIN dim_airport d ON f.dest_airport = d.airport_code GROUP BY o.city, d.city, origin_airport, dest_airport HAVING num_flights >= 365 ORDER BY avg_delay_rate DESC LIMIT 10''').df()
print(con.execute(''' SELECT AVG(delayed) FROM fact_flights''').df())
display(by_route)

   avg(delayed)
0      0.216824


,origin_city,destination_city,origin_airport,dest_airport,num_flights,avg_delay_rate,cancellation_rate
0,Chicago,San Francisco,MDW,SFO,485,0.506276,0.012371
1,Knoxville,Fort Lauderdale,TYS,FLL,481,0.446089,0.008316
2,Dallas/Fort Worth,Harrisburg,DFW,MDT,365,0.426554,0.030137
3,Chicago,Eagle,ORD,EGE,413,0.416244,0.036320
4,Philadelphia,Washington,PHL,DCA,789,0.403718,0.043093
5,Dallas,Charlotte,DAL,CLT,377,0.403226,0.010610
6,Fort Lauderdale,Asheville,FLL,AVL,659,0.401235,0.010622
7,Dallas/Fort Worth,Buffalo,DFW,BUF,578,0.399287,0.029412
8,San Luis Obispo,San Francisco,SBP,SFO,1145,0.392920,0.012227
9,Arcata/Eureka,San Francisco,ACV,SFO,1292,0.390088,0.029412


Among routes with atleast 365 flights, the most delayed was Chicago to San Fransico, late 50.6% of the time, versus 22% across all flights. San Fransisco is the destination for 3 of the top 10 results, and both Chicago airports appear. These are patterns in the data, but the query does not show what is causing them.